# Deception Classifier — Colab GPU Training
Run each cell in order. Make sure the runtime is set to **T4 GPU**:  
Runtime → Change runtime type → T4 GPU

## 1. Clone repo & install dependencies

In [5]:
!git clone https://github.com/Lucca878/deceptionClassifierTraining.git
%cd deceptionClassifierTraining

Cloning into 'deceptionClassifierTraining'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 165 (delta 9), reused 164 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (165/165), 4.30 MiB | 4.65 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/deceptionClassifierTraining/deceptionClassifierTraining


In [6]:
!pip install -q -r requirements.txt

## 2. Verify GPU

In [7]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: NVIDIA A100-SXM4-40GB


## 3. Train DistilBERT

In [8]:
!python src/pipeline/run_pipeline.py \
  --mode train \
  --model distilbert \
  --output_root ./outputs \
  --seed 42

Training model preset: distilbert
Backbone: distilbert-base-uncased
Map: 100% 3633/3633 [00:00<00:00, 6412.90 examples/s]
Map: 100% 909/909 [00:00<00:00, 6333.13 examples/s]
Loading weights: 100% 100/100 [00:00<00:00, 1729.61it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because

## 4. Check output

In [9]:
import os, glob

# Find the latest training run
run_dirs = sorted(glob.glob("outputs/distilbert_*"))
latest = run_dirs[-1] if run_dirs else None
print("Latest run:", latest)

if latest:
    for f in os.listdir(latest):
        print(" ", f)

Latest run: outputs/distilbert_20260423_144743
  full_train_checkpoints
  run_metadata.json
  cv_results.csv
  model
  cv_checkpoints


In [11]:
import pandas as pd
import os

cv_path = os.path.join(latest, "cv_results.csv") if latest else None
if cv_path and os.path.exists(cv_path):
    # File is semicolon-separated
    df = pd.read_csv(cv_path, sep=";")
    print(df.head())
    print("\nRows, cols:", df.shape)

    # Main CV metric from stored predictions
    mean_acc = df["Correct_predictions"].mean()
    print("Mean CV accuracy:", round(mean_acc, 4))

    # Optional per-split and per-class breakdown
    print("\nAccuracy by split:")
    print(df.groupby("Split")["Correct_predictions"].mean().round(4))

    print("\nRecall by label (0=truthful, 1=deceptive):")
    print(df.groupby("Label")["Correct_predictions"].mean().round(4))

   Participant_id    Split  Prediction  Label  Correct_predictions
0             577  split_1           0      1                False
1            1743  split_1           1      0                False
2            2687  split_1           1      0                False
3             219  split_1           1      0                False
4             354  split_1           1      1                 True

Rows, cols: (4542, 5)
Mean CV accuracy: 0.7688

Accuracy by split:
Split
split_1    0.7602
split_2    0.7613
split_3    0.7874
split_4    0.7687
split_5    0.7665
Name: Correct_predictions, dtype: float64

Recall by label (0=truthful, 1=deceptive):
Label
0    0.7257
1    0.8086
Name: Correct_predictions, dtype: float64


## 5. Download the trained model

In [19]:
import os, glob, shutil

# Find the newest trained model folder across possible working dirs
model_dirs = sorted(glob.glob("/content/**/outputs/distilbert_*/model", recursive=True))
if not model_dirs:
    raise FileNotFoundError("No trained model folder found under /content/**/outputs/distilbert_*/model")

model_dir = model_dirs[-1]
zip_base = "/content/distilbert_retrained"
zip_file = zip_base + ".zip"

print("Using model dir:", model_dir)
shutil.make_archive(zip_base, "zip", model_dir)
print("Created zip:", zip_file)
print("Zip exists:", os.path.exists(zip_file), "| Size (MB):", round(os.path.getsize(zip_file) / 1024 / 1024, 2))

Using model dir: /content/deceptionClassifierTraining/deceptionClassifierTraining/outputs/distilbert_20260423_144743/model
Created zip: /content/distilbert_retrained.zip
Zip exists: True | Size (MB): 235.75


In [22]:
from google.colab import drive
import shutil, os

src = "/content/distilbert_retrained.zip"
assert os.path.exists(src), f"Missing file: {src}"

drive.mount("/content/drive")
dst = "/content/drive/MyDrive/distilbert_retrained.zip"
shutil.copy(src, dst)
print("Copied to:", dst)

Mounted at /content/drive
Copied to: /content/drive/MyDrive/distilbert_retrained.zip
